# ਪਾਠ 11 - ਮਾਡਲ ਕੰਟੈਕਸਟ ਪ੍ਰੋਟੋਕੋਲ (MCP)

The **ਮਾਡਲ ਕੰਟੈਕਸਟ ਪ੍ਰੋਟੋਕੋਲ (MCP)** ਇੱਕ ਖੁੱਲਾ ਮਿਆਰ ਹੈ ਜੋ ਏਜੰਟਾਂ ਨੂੰ ਰਨਟਾਈਮ 'ਤੇ ਡਾਇਨੈਮਿਕ ਤੌਰ 'ਤੇ ਟੂਲ, ਸਰੋਤ ਅਤੇ ਡੇਟਾ-ਸਰੋਤ ਖੋਜਣ ਅਤੇ ਵਰਤਣ ਦੇ ਯੋਗ ਬਣਾਉਂਦਾ ਹੈ। ਇੱਕ ਏਜੰਟ ਵਿੱਚ ਟੂਲਾਂ ਨੂੰ ਹਾਰਡਕੋਡ ਕਰਨ ਦੀ ਥਾਂ, MCP ਏਜੰਟਾਂ ਨੂੰ ਬਾਹਰੀ ਸਰਵਰਾਂ ਨਾਲ ਜੁੜਨ ਦੀ ਆਗਿਆ ਦਿੰਦਾ ਹੈ ਜੋ ਮੰਗ 'ਤੇ ਸਮਰੱਥਾਵਾਂ ਪ੍ਰਦਰਸ਼ਿਤ ਕਰਦੇ ਹਨ।

In this lesson, you'll learn:
- MCP ਕੀ ਹੈ ਅਤੇ ਏਜੰਟ ਸਿਸਟਮਾਂ ਲਈ ਇਹ ਕਿਉਂ ਮਹੱਤਵਪੂਰਨ ਹੈ
- MCP ਦਾ ਕਲਾਇੰਟ-ਸਰਵਰ ਆਰਕੀਟੈਕਚਰ ਕਿਵੇਂ ਕੰਮ ਕਰਦਾ ਹੈ
- MCP-ਸ਼ੈਲੀ ਟੂਲ ਖੋਜ ਵਰਤਣ ਵਾਲੇ ਏਜੰਟ ਕਿਵੇਂ ਬਣਾਏ ਜਾ ਸਕਦੇ ਹਨ


## ਸੈਟਅਪ

**ਜਰੂਰੀਆਂ:**
- Azure AI Foundry ਪ੍ਰੋਜੈਕਟ ਜਿਸ ਵਿੱਚ ਇੱਕ ਡਿਪਲੋਇਡ ਮਾਡਲ ਹੋਵੇ
- ਪ੍ਰਮਾਣਿਕਤਾ ਲਈ `az login` ਚਲਾਓ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ਮਾਡਲ ਕੰਟੈਕਸਟ ਪ੍ਰੋਟੋਕੋਲ (MCP) ਕੀ ਹੈ?

MCP ਬਾਹਰੀ ਟੂਲਾਂ ਅਤੇ ਡੇਟਾ ਸਰੋਤਾਂ ਦੀ ਖੋਜ ਕਰਨ ਅਤੇ ਉਨ੍ਹਾਂ ਨਾਲ ਅੰਤਰਕਿਰਿਆ ਕਰਨ ਲਈ ਇੱਕ ਮਿਆਰੀ ਢੰਗ ਨਿਰਧਾਰਤ ਕਰਦਾ ਹੈ:

- **MCP ਸਰਵਰ**: ਇਕ ਮਿਆਰੀ ਪ੍ਰੋਟੋਕੋਲ ਰਾਹੀਂ ਟੂਲ, ਸਰੋਤ ਅਤੇ ਪ੍ਰੌੰਪਟ ਉਪਲਬਧ ਕਰਵਾਉਂਦਾ ਹੈ
- **MCP ਕਲਾਇੰਟ**: ਏਜੰਟ ਰਨਟਾਈਮ ਜੋ ਸਰਵਰਾਂ ਨਾਲ ਜੁੜਦਾ ਹੈ ਅਤੇ ਉਪਲਬਧ ਸਮਰੱਥਾਵਾਂ ਦੀ ਖੋਜ ਕਰਦਾ ਹੈ
- **ਡਾਇਨਾਮਿਕ ਖੋਜ**: ਏਜੰਟਾਂ ਨੂੰ ਹਾਰਡਕੋਡ ਕੀਤੇ ਟੂਲਾਂ ਦੀ ਲੋੜ ਨਹੀਂ — ਉਹ ਰਨਟਾਈਮ 'ਤੇ ਜੋ ਉਪਲਬਧ ਹੈ ਉਹ ਖੋਜ ਲੈਂਦੇ ਹਨ

ਇਹ ਵਿਸ਼ਤਰੀਯੋਗ ਏਜੰਟ ਸਿਸਟਮ ਬਣਾਉਣ ਲਈ ਬਹੁਤ ਪ੍ਰਭਾਵਸ਼ਾਲੀ ਹੈ, ਜਿੱਥੇ ਨਵੀਆਂ ਸਮਰੱਥਾਵਾਂ ਨੂੰ ਏਜੰਟ ਕੋਡ ਨੂੰ ਬਦਲੇ ਬਿਨਾਂ ਜੋੜਿਆ ਜਾ ਸਕਦਾ ਹੈ.


## MCP ਕਿਵੇਂ ਕੰਮ ਕਰਦਾ ਹੈ

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. ਏਜੰਟ (MCP ਕਲਾਇਂਟ) ਇੱਕ MCP ਸਰਵਰ ਨਾਲ ਜੁੜਦਾ ਹੈ
2. ਸਰਵਰ ਉਪਲਬਧ ਟੂਲਾਂ ਅਤੇ ਉਹਨਾਂ ਦੀਆਂ ਸਕੀਮਿਆਂ ਦੀ ਸੂਚੀ ਨਾਲ ਜਵਾਬ ਦਿੰਦਾ ਹੈ
3. ਫਿਰ ਏਜੰਟ ਆਪਣੀ ਤਰਕ-ਵਿਚਾਰ ਦੌਰਾਨ ਮਿਲੇ ਕਿਸੇ ਵੀ ਟੂਲ ਨੂੰ ਕਾਲ ਕਰ ਸਕਦਾ ਹੈ
4. ਨਤੀਜੇ ਉਹੀ ਪ੍ਰੋਟੋਕੋਲ ਰਾਹੀਂ ਵਾਪਸ ਆਉਂਦੇ ਹਨ


## MCP ਟੂਲ ਖੋਜ ਦਾ ਅਨੁਕਰਨ

ਕਿਉਂਕਿ ਇੱਕ ਅਸਲ MCP ਸਰਵਰ ਨੂੰ ਚੱਲ ਰਹੀ ਸਰਵਰ ਪ੍ਰਕਿਰਿਆ ਦੀ ਲੋੜ ਹੁੰਦੀ ਹੈ, ਅਸੀਂ ਉਹ ਪੈਟਰਨ `@tool` ਫੰਕਸ਼ਨਾਂ ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਦਰਸਾਵਾਂਗੇ ਜੋ MCP-ਸੰযুক্ত ਰਹਾਇਸ਼ ਸੇਵਾ ਦੁਆਰਾ ਮੁਹੱਈਆ ਕੀਤੇ ਜਾਣ ਵਾਲੇ ਕੰਮਾਂ ਦੀ ਨਕਲ ਕਰਦੀਆਂ ਹਨ।

ਉਤਪਾਦਨ ਵਿੱਚ, ਇਹ ਟੂਲ ਸਥਾਨਕ ਤੌਰ 'ਤੇ ਪਰਿਭਾਸ਼ਿਤ ਕਰਨ ਦੀ ਬਜਾਏ MCP ਸਰਵਰ ਤੋਂ ਗਤੀਸ਼ੀਲ ਤਰੀਕੇ ਨਾਲ ਖੋਜੇ ਜਾਣਗੇ।


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## MCP-ਸ਼ੈਲੀ ਦੇ ਉਪਕਰਣ ਨਾਲ ਇੱਕ ਏਜੰਟ ਬਣਾਉਣਾ


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP ਉਤਪਾਦਨ ਵਿੱਚ

ਇੱਕ ਉਤਪਾਦਨ ਵਾਤਾਵਰਣ ਵਿੱਚ, MCP ਤਾਕਤਵਰ ਪੈਟਰਨਾਂ ਨੂੰ ਯੋਗ ਬਣਾਉਂਦਾ ਹੈ:

- **ਡਾਇਨਾਮਿਕ ਟੂਲ ਖੋਜ**: ਏਜੰਟ MCP ਸਰਵਰਾਂ ਨਾਲ ਜੁੜਦੇ ਹਨ ਅਤੇ ਰਨਟਾਈਮ 'ਤੇ ਟੂਲ ਲੱਭਦੇ ਹਨ
- **ਵੱਖਰੀ ਬਣਤਰ**: ਟੂਲ ਪ੍ਰਦਾਤਾ ਏਜੰਟ ਤੋਂ ਸਵਤੰਤਰ ਤੌਰ 'ਤੇ ਅਪਡੇਟ ਕਰ ਸਕਦੇ ਹਨ
- **ਸੰਗਠਨ-ਪਾਰ ਸਾਂਝ**: ਟੀਮਾਂ MCP ਸਰਵਰਾਂ ਰਾਹੀਂ ਸਮਰੱਥਾਵਾਂ ਪ੍ਰਗਟ ਕਰ ਸਕਦੀਆਂ ਹਨ, ਜਿਨ੍ਹਾਂ ਨੂੰ ਕੋਈ ਵੀ ਏਜੰਟ ਵਰਤ ਸਕਦਾ ਹੈ
- **Microsoft Agent Framework ਸਹਿਯੋਗ**: MAF ਕੋਲ `mcp` ਇੰਟੀਗ੍ਰੇਸ਼ਨ ਰਾਹੀਂ ਬਿਲਟ-ਇਨ MCP ਕਲਾਇੰਟ ਸਹਿਯੋਗ ਹੈ

MAF ਨਾਲ ਇੱਕ ਅਸਲ MCP ਸਰਵਰ ਵਰਤਣ ਲਈ, ਤੁਸੀਂ `hosted_mcp_tool()` ਜਾਂ MCP ਕਲਾਇੰਟ ਇੰਟੀਗ੍ਰੇਸ਼ਨ ਰਾਹੀਂ ਜੁੜੋਗੇ।

**ਹੋਰ ਜਾਣੋ:**
- [MCP ਵਿਸ਼ੇਸ਼ਣ](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP ਸਹਿਯੋਗ](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## ਸਾਰ

ਇਸ ਪਾਠ ਵਿੱਚ, ਤੁਸੀਂ ਸਿੱਖਿਆ:
- **MCP** ਏਜਨਟਾਂ ਅਤੇ ਟੂਲ ਪ੍ਰਦਾਤਿਆਂ ਦੇ ਵਿਚਕਾਰ ਗਤੀਸ਼ੀਲ ਟੂਲ ਖੋਜ ਲਈ ਇੱਕ ਖੁੱਲਾ ਮਿਆਰ ਹੈ
- **ਕਲਾਇੰਟ-ਸਰਵਰ ਸੰਰਚਨਾ** ਏਜਨਟਾਂ ਨੂੰ ਚਲਾਉਣ ਸਮੇਂ ਯੋਗਤਾਵਾਂ ਦਾ ਪਤਾ ਲਗਾਉਣ ਦਿੰਦੀ ਹੈ
- MCP ਵਿਸਤਾਰਯੋਗ ਅਤੇ ਵਿਛੁੜੀਆਂ ਏਜਨਟ ਪ੍ਰਣਾਲੀਆਂ ਨੂੰ ਯੋਗ ਬਣਾਉਂਦਾ ਹੈ, ਜਿੱਥੇ ਕੋਡ ਬਦਲਾਅ ਕੀਤੇ ਬਿਨਾਂ ਟੂਲ ਜੋੜੇ ਜਾ ਸਕਦੇ ਹਨ
- Microsoft Agent Framework ਉਤਪਾਦਨ ਵਰਤੋਂ ਲਈ **ਅੰਦਰੂਨੀ MCP ਸਹਾਇਤਾ** ਪ੍ਰਦਾਨ ਕਰਦਾ ਹੈ


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
ਅਸਵੀਕਾਰ:
ਇਸ ਦਸਤਾਵੇਜ਼ ਦਾ ਅਨੁਵਾਦ ਏ.ਆਈ. ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਿਵੇਂ ਕਿ ਅਸੀਂ ਸਹੀਤਾ ਲਈ ਯਤਨ ਕਰਦੇ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਦਿਆਨ ਰੱਖੋ ਕਿ ਆਟੋਮੇਟਿਕ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਨੁਚਿੱਤਤਾਵਾਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਪਾਇਆ ਗਿਆ ਦਸਤਾਵੇਜ਼ ਹੀ ਅਧਿਕਾਰਿਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਮਹੱਤਵਪੂਰਨ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫ਼ਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੀ ਵਰਤੋਂ ਕਾਰਨ ਪੈਦਾਅ ਹੋਣ ਵਾਲੀ ਕਿਸੇ ਵੀ ਗਲਤਫ਼ਹਮੀ ਜਾਂ ਭੁੱਲ ਲਈ ਜ਼ਿੰਮੇਵਾਰ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
